In [3]:
import requests, sys
sys.path.append('backend/app/')
import pandas as pd, geopandas as gpd
from backend.app.services import functions

In [8]:
CLIENT_ID = "xyWh5HlljU29alKybzYHQA=="
url = "https://hydapi.nve.no/api/v1"
url_station = f"{url}/Stations"

In [13]:
headers = {
    "X-API-Key": CLIENT_ID
}
response = requests.get(url_station, headers=headers)
response.raise_for_status()
stations = response.json()
df = pd.DataFrame(stations['data'])

In [14]:
df.head()

,stationId,stationName,latitude,longitude,utmEast_Z33,utmNorth_Z33,masl,riverName,councilNumber,councilName,...,culQ5,culQ10,culQ20,culQ50,culHm,culH5,culH10,culH20,culH50,seriesList
0,1.15.0,Femsjø,59.13015,11.48516,298887,6559841,81,Haldenvassdraget,3101,Halden,...,NaN,NaN,NaN,NaN,79.5376,79.6761,79.8303,79.9981,80.2486,"[{'parameterName': 'Vannstand', 'parameter': 1..."
1,1.200.0,Lierelv,59.92038,11.53180,306146,6647624,134,Haldenvassdraget,3226,Aurskog-Høland,...,32.6537,34.9251,36.4982,37.9152,2.2229,2.3076,2.3381,2.3585,2.3764,"[{'parameterName': 'Vannstand', 'parameter': 1..."
2,1.41.0,Lognselva,59.80011,11.44331,300484,6634505,148,Haldenvassdraget,3226,Aurskog-Høland,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[{'parameterName': 'Vannstand', 'parameter': 1..."
3,1.42.0,Øymarksjø,59.33290,11.63712,308715,6581952,109,Haldenvassdraget,3124,Aremark,...,NaN,NaN,NaN,NaN,108.7285,108.9429,109.1298,109.3144,109.5614,"[{'parameterName': 'Vannstand', 'parameter': 1..."
4,1.43.0,Aspern og Ara,59.14667,11.69273,310848,6561073,106,Haldenvassdraget,3124,Aremark,...,NaN,NaN,NaN,NaN,106.0582,106.1430,106.2918,106.4791,106.8088,"[{'parameterName': 'Vannstand', 'parameter': 1..."


In [17]:
df1 = df.explode('seriesList', ignore_index=True)
obs = pd.json_normalize(df1['seriesList'])
obs.head()

,parameterName,parameter,versionNo,unit,serieFrom,serieTo,resolutionList
0,Vannstand,1000,2,m,1939-07-01T00:00:00,NaN,"[{'resTime': 0, 'method': 'Instantaneous', 'ti..."
1,Magasinvolum,1004,2,millioner m³,NaN,NaN,"[{'resTime': 0, 'method': 'Instantaneous', 'ti..."
2,Vannstand,1000,1,m,2010-09-16T00:00:00,NaN,"[{'resTime': 0, 'method': 'Instantaneous', 'ti..."
3,Vannføring,1001,1,m³/s,NaN,NaN,"[{'resTime': 0, 'method': 'Instantaneous', 'ti..."
4,Vanntemperatur,1003,1,°C,2010-01-01T00:00:00,NaN,"[{'resTime': 0, 'method': 'Instantaneous', 'ti..."


In [35]:
df_wide = df.pivot_table(
    index=["referenceTime", "sourceId"],
    columns="elementId",
    values="value",
    aggfunc="first"
).reset_index()
df_wide

elementId,referenceTime,sourceId,air_temperature,sum(precipitation_amount PT1H)
0,2026-09-01 00:00:00+00:00,SN99950:0,3.7,0.0
1,2026-09-01 01:00:00+00:00,SN99950:0,3.5,0.0
2,2026-09-01 02:00:00+00:00,SN99950:0,3.6,0.0
3,2026-09-01 03:00:00+00:00,SN99950:0,3.6,0.0
4,2026-09-01 04:00:00+00:00,SN99950:0,3.4,0.0
...,...,...,...,...
715,2026-09-30 19:00:00+00:00,SN99950:0,8.3,2.3
716,2026-09-30 20:00:00+00:00,SN99950:0,8.3,2.6
717,2026-09-30 21:00:00+00:00,SN99950:0,8.5,0.1
718,2026-09-30 22:00:00+00:00,SN99950:0,8.7,0.0


In [20]:
params = {
    "sources": "SN99950", "elements": 'air_temperature',
    "referencetime": "2026-09-01/2026-10-01",
}
response = requests.get(url, params=params, auth=(CLIENT_ID, ""))
# response.raise_for_status()
data = response.json()
stations = pd.DataFrame(data['data'])
stations

,sourceId,referenceTime,observations
0,SN99950:0,2026-09-01T00:00:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.7..."
1,SN99950:0,2026-09-01T00:10:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.5..."
2,SN99950:0,2026-09-01T00:20:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.3..."
3,SN99950:0,2026-09-01T00:30:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.5..."
4,SN99950:0,2026-09-01T00:40:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.5..."
...,...,...,...
4315,SN99950:0,2026-09-30T23:10:00.000Z,"[{'elementId': 'air_temperature', 'value': 8.9..."
4316,SN99950:0,2026-09-30T23:20:00.000Z,"[{'elementId': 'air_temperature', 'value': 8.8..."
4317,SN99950:0,2026-09-30T23:30:00.000Z,"[{'elementId': 'air_temperature', 'value': 8.9..."
4318,SN99950:0,2026-09-30T23:40:00.000Z,"[{'elementId': 'air_temperature', 'value': 8.9..."


In [25]:
stations['observations'][0]

[{'elementId': 'air_temperature',
  'value': 3.7,
  'unit': 'degC',
  'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2},
  'timeOffset': 'PT0H',
  'timeResolution': 'PT1H',
  'timeSeriesId': 0,
  'performanceCategory': 'C',
  'exposureCategory': '1',
  'qualityCode': 0},
 {'elementId': 'air_temperature',
  'value': 3.7,
  'unit': 'degC',
  'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2},
  'timeOffset': 'PT0H',
  'timeResolution': 'PT10M',
  'timeSeriesId': 0,
  'performanceCategory': 'C',
  'exposureCategory': '1',
  'qualityCode': 0}]

In [26]:
stations['observations'][1]

[{'elementId': 'air_temperature',
  'value': 3.5,
  'unit': 'degC',
  'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2},
  'timeOffset': 'PT0H',
  'timeResolution': 'PT10M',
  'timeSeriesId': 0,
  'performanceCategory': 'C',
  'exposureCategory': '1',
  'qualityCode': 0}]

In [27]:
stations['observations'][2]

[{'elementId': 'air_temperature',
  'value': 3.3,
  'unit': 'degC',
  'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2},
  'timeOffset': 'PT0H',
  'timeResolution': 'PT10M',
  'timeSeriesId': 0,
  'performanceCategory': 'C',
  'exposureCategory': '1',
  'qualityCode': 0}]

In [ ]:
pd.DataFrame(stations)['observations'].iloc[0]

In [ ]:
import requests

url = "https://frost.met.no/observations/availableTimeSeries/v0.jsonld"

params = {
    "sources": "SN47230"
}

response = requests.get(
    url,
    params=params,
    auth=(CLIENT_ID, "")
)

response.raise_for_status()

data = response.json()

print(data.keys())

In [ ]:
for item in data["data"]:
    print(
        "Source:", item.get("sourceId"),
        "| Element:", item.get("elementId"),
        "| Unit:", item.get("unit"),
        "| Resolution:", item.get("timeResolution"),
        "| From:", item.get("validFrom"),
        "| To:", item.get("validTo")
    )

In [ ]:
df.columns

In [ ]:
df['geometry'] = df['geometry'].apply(lambda x: x['coordinates'] if isinstance(x, dict) and 'coordinates' in x else None)

In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['geometry'].apply(lambda x: x[0] if x else None), df['geometry'].apply(lambda x: x[1] if x else None)), crs='EPSG:4326')

In [ ]:
gdf

In [ ]:
gdf.to_file("met_stations.geojson", driver='GeoJSON')